# Companion notebook (ខ្មែរ) — `ch05/14_ch05_with_other_llms/ch05-qwen3.ipynb`

> **ប្រភព និងអាជ្ញាបណ្ណ** — កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង
> [`ch05/14_ch05_with_other_llms/ch05-qwen3.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/14_ch05_with_other_llms/ch05-qwen3.ipynb)
> នៃឃ្លាំង [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)
> (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។ Comments ក្នុងកូដត្រូវបានបកប្រែជាខ្មែរដោយសហគមន៍។
> Notebook នេះ **មិនមានអត្ថបទសៀវភៅទេ** (អត្ថបទសៀវភៅជាកម្មសិទ្ធិ Manning ដកចេញពីអាជ្ញាបណ្ណបើកចំហ) —
> សម្រាប់ការពន្យល់សូមមើលសៀវភៅផ្លូវការ ឬ notebook ដើម។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតដើមរបស់វា ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការពីឃ្លាំង upstream។

In [ ]:
# pip install tokenizers

In [ ]:
from importlib.metadata import version

pkgs = [
    "matplotlib", 
    "numpy", 
    "torch",
    "tokenizers",       # to implement the Qwen3 tokenizer
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
######################
### Qwen3 Code
######################

import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)


class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale

        if self.shift is not None:
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)


def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half
    x2 = x[..., head_dim // 2 :]  # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)


class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # Attention
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)

        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin):
        # ការតភ្ជាប់កាត់ (shortcut) សម្រាប់ប្លុក attention
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # រូបរាង [batch_size, num_tokens, emb_size]
        x = x + shortcut  # ដាក់បញ្ចូលដើមត្រឡប់មកវិញ

        # ការតភ្ជាប់កាត់ (shortcut) សម្រាប់ប្លុក feed-forward
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # ដាក់បញ្ចូលដើមត្រឡប់មកវិញ

        return x


class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Main model parameters
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(  # ModuleList since Sequential can only accept one input, and we need `x, mask, cos, sin`
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # Uncomment the following code to tie weights
        # self.out_head.weight = self.tok_emb.weight
        # torch.nn.init.normal_(self.out_head.weight, mean=0.0, std=0.02)

        # Reusable utilities
        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg


    def forward(self, in_idx):
        # ដំណើរការ forward
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)
        
        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

In [ ]:
#######################
### Initialize Qwen3 
#######################

# 0.6B model
QWEN3_CONFIG = {
    "vocab_size": 151_936,  # ទំហំ vocabulary
    "context_length": 40_960,        # Context length that was used to train the model
    "emb_dim": 1024,  # វិមាត្រ embedding
    "n_heads": 16,  # ចំនួន attention heads
    "n_layers": 28,  # ចំនួនស្រទាប់
    "hidden_dim": 3072,              # Size of the intermediate dimension in FeedForward
    "head_dim": 128,                 # Size of the heads in GQA
    "qk_norm": True,                 # Whether to normalize queries and keys in GQA
    "n_kv_groups": 8,                # Key-Value groups for grouped-query attention
    "rope_base": 1_000_000.0,        # The base in RoPE's "theta"
    "dtype": torch.bfloat16,         # Lower-precision dtype to reduce memory usage
}

QWEN3_CONFIG["train_context_length"] = 256  # It's a small dataset, and we also want to keep memory usage reasonable

torch.manual_seed(123)
model = Qwen3Model(QWEN3_CONFIG)
model.eval();

In [ ]:
#######################
### Set up tokenizer
#######################

import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>"
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

repo_id = "Qwen/Qwen3-0.6B-Base"
tokenizer_file_path = "tokenizer.json"
local_dir = "."

hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_file_path,
    repo_id=repo_id,
    apply_chat_template=False,
    add_generation_prompt=False,
    add_thinking=False
)

In [ ]:
# Same as chapter 4

def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # កាត់ context បច្ចុបន្ន បើលើសពីទំហំ context ដែលគាំទ្រ
        # ឧទាហរណ៍ បើ LLM គាំទ្រតែ 5 tokens ហើយទំហំ context គឺ 10 នោះមានតែ 5 tokens ចុងក្រោយប៉ុណ្ណោះដែលត្រូវបានប្រើជា context
        idx_cond = idx[:, -context_size:]

        # យកការព្យាករណ៍
        with torch.no_grad():
            logits = model(idx_cond)

        # ផ្តោតតែលើ time step ចុងក្រោយ
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # បន្ថែម idx ដែលបានស្ទូចយក ទៅបន្សំកំពុងដំណើរការ
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # បន្ថែមវិមាត្រ batch
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # យកវិមាត្រ batch ចេញ
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=QWEN3_CONFIG["train_context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [ ]:
# តួអក្សរ ៩៩ ដំបូង
print(text_data[:99])

In [ ]:
# តួអក្សរ ៩៩ ចុងក្រោយ
print(text_data[-99:])

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # បំបែកអត្ថបទទាំងមូលជា token
        token_ids = tokenizer.encode(txt)

        # ប្រើ sliding window ដើម្បីកាត់អត្ថបទជាបន្សំត្រួតគ្នា ប្រវែង max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# Note that we have to change the function below because we previously hard-coded the
# GPT-2 tokenizer in the data loader
def create_dataloader_v1(txt, tokenizer, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # ចាប់ផ្តើម tokenizer
    # tokenizer = tiktoken.get_encoding("gpt2")
    tokenizer = tokenizer

    # បង្កើត dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # បង្កើត dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


# សមាមាត់ train/validation
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    tokenizer=tokenizer,
    batch_size=2,
    max_length=QWEN3_CONFIG["train_context_length"],
    stride=QWEN3_CONFIG["train_context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    tokenizer=tokenizer,
    batch_size=2,
    max_length=QWEN3_CONFIG["train_context_length"],
    stride=QWEN3_CONFIG["train_context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # បន្ថយចំនួន batches ឱ្យស្ើនឹងចំនួន batches សរុបក្នុង data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # ប្រើ PyTorch 2.9 ឬថ្មីជាងនេះ សម្រាប់លទ្ធផល mps ស្ថិរ
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")


print(f"Using {device} device.")


model.to(device)


torch.manual_seed(123)  # សម្រាប់ភាពស្របគ្នា ដោយសារការច្របល់ក្នុង data loader

with torch.no_grad():  # បិទការតាមដាន gradient សម្រាប់ប្រសិទ្ធភាព ព្រោះយើងមិនកំពុងបណ្តុះបណ្តាលទេ
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # ចាប់ផ្តើមបញ្ជីដើម្បីតាមដាន losses និងចំនួន tokens ដែលបានឃើញ
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # រង្វិលបណ្តុះបណ្តាលមេ
    for epoch in range(num_epochs):
        model.train()  # ដាក់ម៉ូដែលក្នុងរបៀបបណ្តុះបណ្តាល
        
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()  # កំណត់ gradients នៃ loss ពី iteration មុនឡើងវិញ
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()  # គណនា gradients នៃ loss
            optimizer.step()  # ធ្វើបច្ចុបបន្នភាព weights របស់ម៉ូដែលដោយប្រើ gradients នៃ loss
            tokens_seen += input_batch.numel()
            global_step += 1

            # ជំហានវាយតម្លៃជាជម្រើស
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # បង្ហាញអត្ថបទគំរូបន្ទាប់ពី epoch នីមួយៗ
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.cfg["context_length"]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # ទ្រង់ទ្រាយបង្ហាញសង្ខេប
    model.train()

In [ ]:
# Note:
# ដោះ comment កូដខាងក្រោមដើម្បីគណនារយៈពេលដំណើរការ
# import time
# start_time = time.time()

torch.manual_seed(123)
model = Qwen3Model(QWEN3_CONFIG)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 40
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

# Note:
# ដោះ comment កូដខាងក្រោមដើម្បីបង្ហាញរយៈពេលដំណើរការ
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # គូសក្រាផ loss នៃ train និង validation ធ្នឹងនឹង epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # បង្ហាញតែស្លាកជាចំនួនគត់លើអ័ក្ស x

    # បង្កើតអ័ក្ស x ទីពីរសម្រាប់ចំនួន tokens ដែលបានឃើញ
    ax2 = ax1.twiny()  # បង្កើតអ័ក្ x ទីពីរដែលចែករំលែកអ័ក្ស y តែមួយ
    ax2.plot(tokens_seen, train_losses, alpha=0)  # ក្រាផមើលមិនឃើញសម្រាប់តម្រឹម ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # លៃតម្រូវ layout ដើម្បីទុកកន្លែង
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

In [ ]:
inference_device = torch.device("cpu")

model.to(inference_device)
model.eval()

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=QWEN3_CONFIG["train_context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Painting", tokenizer).to(inference_device),
    max_new_tokens=25,
    context_size=QWEN3_CONFIG["train_context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ